# 15 — Medical DPO with Labeled Preference Data

This notebook applies **Direct Preference Optimisation (DPO)** to the SFT model using
preference pairs derived directly from MedQA's labeled answer choices.

**Pipeline:**
1. Understand why medical DPO is different from standard DPO
2. Build preference pairs from MedQA (correct answer = chosen, wrong options = rejected)
3. Load the SFT checkpoint and a frozen reference copy
4. Implement the DPO loss
5. Train for 1000 steps
6. Compare MCQ accuracy: SFT vs DPO

## 2 — Why Medical DPO is Different

### Standard DPO Pipeline

In the standard RLHF/DPO workflow:
1. Generate multiple candidate responses for each prompt using the SFT model
2. Have human annotators (or a reward model) rank the responses
3. Form (chosen, rejected) pairs from the rankings
4. Train with DPO to make `p(chosen)` > `p(rejected)` relative to the reference model

This is expensive: it requires running inference to generate candidates, then labelling.
The quality of the preference signal depends heavily on the quality of generation and annotation.

### The Medical DPO Insight

USMLE-style MCQs already provide a **gold-labeled preference structure**:
- The **correct answer** is the unambiguously chosen response
- The **wrong options** (A, B, C, D minus the correct one) are unambiguously rejected
- These labels were written and validated by medical educators — far higher quality than
  typical crowd-sourced preference annotations

This means we can form high-quality (prompt, chosen, rejected) triples without any
generation step. Each question yields up to 3 preference pairs (one per wrong option).

**Why this is a genuine insight:**  
Standard DPO datasets (Anthropic HH, Stanford SHP) use heuristic or crowd-sourced labels
that can be noisy. Medical MCQ labels are unambiguous by construction — the correct answer
is factually verifiable. This makes medical DPO cleaner and more reliable than typical DPO applications.

**Reference:**  
Rafailov et al., "Direct Preference Optimization: Your Language Model is Secretly a Reward Model", NeurIPS 2023.

## 1 — Setup

In [ ]:
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3 — Build Preference Pairs

We call `build_dpo_pairs_from_medqa()` from `datasets/medical.py` to construct the
preference dataset.

Each pair consists of:
- `prompt`: the question formatted as an instruction
- `chosen`: the correct answer option (letter + explanation)
- `rejected`: one of the wrong answer options

With `max_pairs=8000` and 3 wrong options per question, this requires at least ~2700 questions.

The pairs are saved to `cfg.MED_DPO_DATASET` as a JSON file for reproducibility.

In [ ]:
from datasets import load_dataset
import config as cfg

raw_medqa = load_dataset(cfg.MED_MEDQA_DATASET, split="train")

# Use only the SFT training range — keep test set (cfg.MED_MEDQA_TRAIN_SIZE onward) clean
train_examples = list(raw_medqa.select(range(min(cfg.MED_MEDQA_TRAIN_SIZE, len(raw_medqa)))))
print(f"Using {len(train_examples):,} MedQA training examples for DPO pair construction")

In [ ]:
from loaders.medical import build_dpo_pairs_from_medqa
from dpo.dataset_generation import save_dpo_dataset
import config as cfg

MAX_PAIRS = 8000

pairs = build_dpo_pairs_from_medqa(train_examples, max_pairs=MAX_PAIRS)
print(f"Built {len(pairs):,} preference pairs")

# Save to disk for reproducibility
save_dpo_dataset(pairs, cfg.MED_DPO_DATASET)
print(f"Saved to: {cfg.MED_DPO_DATASET}")

In [ ]:
# Inspect 3 example pairs to verify the structure
print("=" * 70)
for i in range(min(3, len(pairs))):
    pair = pairs[i]
    print(f"\n--- Pair {i+1} ---")
    print(f"PROMPT (first 200 chars):")
    print(pair.get("prompt", "")[:200])
    print(f"\nCHOSEN:")
    print(str(pair.get("chosen", ""))[:200])
    print(f"\nREJECTED:")
    print(str(pair.get("rejected", ""))[:200])
    print("=" * 70)

print(f"\nPair keys: {list(pairs[0].keys()) if pairs else 'N/A'}")

## 4 — Load SFT Checkpoint

DPO requires two models:
1. **Policy model** (`model`) — the model being trained, initialised from SFT weights
2. **Reference model** (`ref_model`) — a frozen copy of the SFT model

The reference model anchors the KL divergence term in the DPO objective — it prevents
the policy from drifting too far from the SFT model's distribution. Without this anchor,
the model could increase `log p(chosen)` by simply collapsing to a degenerate distribution.

**Memory note:** Both models must be in memory simultaneously during DPO training.
On a 16GB T4, two 25M-parameter models (~200MB each in fp32) fit comfortably.
For models >1B parameters, consider using LoRA for the policy while keeping the
reference frozen (see PEFT notebooks).

In [ ]:
import copy
import torch
from model.gpt import GPT, GPTConfig
from tokenizer.preprocess import load_tokenizer
import config as cfg

MED_BLOCK_SIZE = 512

gpt_config = GPTConfig(
    vocab_size=cfg.VOCAB_SIZE,
    block_size=MED_BLOCK_SIZE,
    n_layer=cfg.N_LAYERS,
    n_head=cfg.N_HEADS,
    n_embd=cfg.D_MODEL,
)

# Load SFT weights into the policy model
model = GPT(gpt_config).to(device)
state = torch.load(cfg.MED_SFT_FINAL_CKPT, map_location=device)
if isinstance(state, dict) and "model" in state:
    state = state["model"]
model.load_state_dict(state)

# Deep copy as the frozen reference model
ref_model = copy.deepcopy(model).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)   # reference model is never updated

med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

print(f"Policy model    : {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params (trainable)")
print(f"Reference model : {sum(p.numel() for p in ref_model.parameters()) / 1e6:.2f}M params (frozen)")

## 5 — DPO Loss

### The DPO Objective

DPO directly optimises the policy without a separate reward model.
The loss for a single (prompt, chosen, rejected) triple is:

$$
\mathcal{L}_{\text{DPO}} = -\log \sigma\!\Big(
    \beta \cdot \underbrace{\big(\log \pi_{\theta}(y_w | x) - \log \pi_{\text{ref}}(y_w | x)\big)}_{\text{chosen advantage}}
    - \beta \cdot \underbrace{\big(\log \pi_{\theta}(y_l | x) - \log \pi_{\text{ref}}(y_l | x)\big)}_{\text{rejected advantage}}
\Big)
$$

Where:
- $\pi_{\theta}$ — the policy model (being trained)
- $\pi_{\text{ref}}$ — the frozen reference model (SFT weights)
- $y_w$ — chosen response (correct answer)
- $y_l$ — rejected response (wrong answer)
- $x$ — the prompt
- $\beta$ — temperature controlling how tightly we stay near the reference

**Intuition:**  
The loss is minimised when the policy assigns higher log-probability to the chosen response
*relative to the reference* than it does to the rejected response. The $\beta$ parameter
controls the strength of the preference — higher $\beta$ forces larger margins.

In [ ]:
import torch
import torch.nn.functional as F
import config as cfg

def compute_sequence_log_prob(model, input_ids, response_start_idx):
    """Compute the mean log-probability of response tokens given the prompt.

    Only response tokens (positions >= response_start_idx) contribute to the
    log-probability. This matches the likelihood-scoring approach used in evaluation.

    Args:
        model             : GPT model (policy or reference)
        input_ids         : (1, T) LongTensor — prompt + response token IDs
        response_start_idx: int — index in input_ids where the response begins

    Returns:
        torch.Tensor scalar: mean log-prob of response tokens
    """
    logits    = model(input_ids[:, :-1])          # (1, T-1, vocab)
    log_probs = F.log_softmax(logits, dim=-1)     # (1, T-1, vocab)
    targets   = input_ids[:, 1:]                  # (1, T-1)

    # Gather log-prob of the actual token at each position
    token_log_probs = log_probs.gather(
        dim=-1,
        index=targets.unsqueeze(-1)
    ).squeeze(-1)   # (1, T-1)

    # Mask to response positions only
    start = max(0, response_start_idx - 1)   # -1 because we predict pos[i] from pos[i-1]
    response_log_probs = token_log_probs[:, start:]

    return response_log_probs.mean()


def dpo_loss(model, ref_model, prompt_ids, chosen_ids, rejected_ids, beta):
    """Compute the DPO loss for a single (prompt, chosen, rejected) triple.

    Implements the loss from Rafailov et al. (2023) Equation 7:
        L = -log sigma(beta * (log pi(chosen|x) - log pi_ref(chosen|x))
                       - beta * (log pi(rejected|x) - log pi_ref(rejected|x)))

    Args:
        model      : policy GPT model (trainable)
        ref_model  : reference GPT model (frozen)
        prompt_ids : list of int, encoded prompt
        chosen_ids : list of int, encoded chosen response
        rejected_ids: list of int, encoded rejected response
        beta       : float, preference temperature (cfg.MED_DPO_BETA)

    Returns:
        torch.Tensor scalar: DPO loss value
    """
    prompt_len = len(prompt_ids)

    # Build input tensors: prompt + response concatenated
    chosen_input   = torch.tensor(prompt_ids + chosen_ids,   dtype=torch.long, device=device).unsqueeze(0)
    rejected_input = torch.tensor(prompt_ids + rejected_ids, dtype=torch.long, device=device).unsqueeze(0)

    # Truncate to block_size to avoid OOM on long sequences
    chosen_input   = chosen_input[:, :MED_BLOCK_SIZE]
    rejected_input = rejected_input[:, :MED_BLOCK_SIZE]

    # Policy log-probs
    policy_chosen_logp   = compute_sequence_log_prob(model,     chosen_input,   prompt_len)
    policy_rejected_logp = compute_sequence_log_prob(model,     rejected_input, prompt_len)

    # Reference log-probs (no grad — reference is frozen)
    with torch.no_grad():
        ref_chosen_logp   = compute_sequence_log_prob(ref_model, chosen_input,   prompt_len)
        ref_rejected_logp = compute_sequence_log_prob(ref_model, rejected_input, prompt_len)

    # Compute log-probability ratios (advantages relative to reference)
    chosen_advantage   = policy_chosen_logp   - ref_chosen_logp
    rejected_advantage = policy_rejected_logp - ref_rejected_logp

    # DPO objective: encourage chosen advantage > rejected advantage
    loss = -F.logsigmoid(beta * (chosen_advantage - rejected_advantage))
    return loss


print("DPO loss function defined.")
print(f"Beta (preference temperature): {cfg.MED_DPO_BETA}")

## 6 — DPO Training Loop

**Training details:**
- 1,000 steps — DPO requires far fewer steps than pretraining or SFT because the model
  already knows the answer format; we are only shifting probability mass
- Small LR (`cfg.MED_DPO_LR = 1e-5`) — DPO is sensitive to large updates
- Small batch (`cfg.MED_DPO_BATCH_SIZE = 8`) — each step processes one pair at a time
  within the batch; memory is doubled vs SFT because we hold both policy and reference
- Checkpoint every 200 steps

**Expected behavior:**
- Loss should start around 0.6–0.7 (near `log(2)` = 0.693, meaning random preference)
- Loss should decrease to ~0.3–0.4 as the model learns to prefer correct answers
- MCQ accuracy should improve 2–5 percentage points above SFT baseline

In [ ]:
import random
import config as cfg

# Shuffle pairs for training
random.shuffle(pairs)
pair_idx = [0]   # mutable reference so we can update inside get_dpo_pair

def get_dpo_pair():
    """Return the next DPO (prompt, chosen, rejected) triple from the shuffled dataset.

    Wraps around when all pairs have been seen — effectively an infinite iterator.

    Returns:
        tuple: (prompt_str, chosen_str, rejected_str)
    """
    idx = pair_idx[0] % len(pairs)
    pair = pairs[idx]
    pair_idx[0] += 1
    return pair.get("prompt", ""), pair.get("chosen", ""), pair.get("rejected", "")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.MED_DPO_LR,
    weight_decay=0.01,
    betas=(0.9, 0.95),
)

print(f"DPO optimizer ready. LR={cfg.MED_DPO_LR:.1e}, beta={cfg.MED_DPO_BETA}")

In [ ]:
# Resume logic — same pattern as DAPT and SFT
DPO_CKPT_PATH = str(cfg.MED_DPO_FINAL_CKPT).replace(".pt", "_ckpt.pt")
start_step = 0
if os.path.exists(DPO_CKPT_PATH):
    print(f"Resuming DPO from: {DPO_CKPT_PATH}")
    ckpt = torch.load(DPO_CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"]
    print(f"  → step {start_step:,}")
else:
    print("Starting DPO from step 0.")

In [ ]:
%%time
# ── DPO Training Loop ─────────────────────────────────────────────────────
model.train()
dpo_loss_log = []

for step in range(start_step, cfg.MED_DPO_MAX_STEPS):

    optimizer.zero_grad()
    batch_loss = torch.tensor(0.0, device=device)

    # Accumulate loss over cfg.MED_DPO_BATCH_SIZE pairs
    # We process one pair at a time and average (memory-efficient vs stacking)
    for _ in range(cfg.MED_DPO_BATCH_SIZE):
        prompt_str, chosen_str, rejected_str = get_dpo_pair()
        if not chosen_str or not rejected_str:
            continue

        prompt_ids   = med_tok.encode(prompt_str).ids
        chosen_ids   = med_tok.encode(str(chosen_str)).ids
        rejected_ids = med_tok.encode(str(rejected_str)).ids

        loss = dpo_loss(
            model, ref_model,
            prompt_ids, chosen_ids, rejected_ids,
            beta=cfg.MED_DPO_BETA,
        )
        batch_loss = batch_loss + loss / cfg.MED_DPO_BATCH_SIZE

    batch_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 20 == 0:
        dpo_loss_log.append((step, batch_loss.item()))
        print(f"step {step:>5} | dpo loss {batch_loss.item():.4f}")

    if step % 200 == 0 and step > 0:
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step},
            DPO_CKPT_PATH,
        )
        print(f"  Checkpoint → {DPO_CKPT_PATH}")

torch.save(model.state_dict(), cfg.MED_DPO_FINAL_CKPT)
print(f"\nDPO complete. Model saved → {cfg.MED_DPO_FINAL_CKPT}")

In [ ]:
# Plot DPO loss curve
if dpo_loss_log:
    try:
        import matplotlib.pyplot as plt
        steps  = [s for s, _ in dpo_loss_log]
        losses = [l for _, l in dpo_loss_log]
        plt.figure(figsize=(9, 3))
        plt.plot(steps, losses, linewidth=1.5, color="darkorange")
        plt.axhline(0.693, color="gray", linestyle="--", alpha=0.5, label="log(2) — random preference")
        plt.xlabel("Step")
        plt.ylabel("DPO Loss")
        plt.title("DPO Training Loss — Medical MedQA Preferences")
        plt.legend()
        plt.tight_layout()
        plt.show()
    except ImportError:
        print(f"{'Step':>8}  {'DPO Loss':>12}")
        for s, l in dpo_loss_log:
            print(f"{s:>8}  {l:>12.4f}")

## 7 — Does DPO Help?

We compare MCQ accuracy for the SFT and DPO models on the same held-out test set
(last 200 examples from the MedQA test split defined in notebook 14).

**Expected improvement:** 2–5 percentage points. DPO is a refinement step, not a dramatic
re-training — it shifts probability mass toward correct answers while preserving the
model's overall language quality.

In [ ]:
from datasets import load_dataset
from evaluation.medical_metrics import evaluate_mcq_accuracy
import config as cfg

# Load test examples (same slice as notebook 14)
raw_medqa     = load_dataset(cfg.MED_MEDQA_DATASET, split="train")
TEST_SIZE     = 200
test_start    = cfg.MED_MEDQA_TRAIN_SIZE
test_examples = list(raw_medqa.select(range(test_start, min(test_start + TEST_SIZE, len(raw_medqa)))))

print(f"Evaluating on {len(test_examples)} test examples...")

# ── SFT accuracy ──────────────────────────────────────────────────────────
sft_state = torch.load(cfg.MED_SFT_FINAL_CKPT, map_location=device)
if isinstance(sft_state, dict) and "model" in sft_state:
    sft_state = sft_state["model"]
from model.gpt import GPT, GPTConfig
sft_model = GPT(GPTConfig(
    vocab_size=cfg.VOCAB_SIZE, block_size=MED_BLOCK_SIZE,
    n_layer=cfg.N_LAYERS, n_head=cfg.N_HEADS, n_embd=cfg.D_MODEL,
)).to(device)
sft_model.load_state_dict(sft_state)
sft_model.eval()

sft_acc = evaluate_mcq_accuracy(
    model=sft_model, tokenizer=med_tok, device=device,
    examples=test_examples, max_examples=TEST_SIZE, verbose=False,
)
del sft_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"SFT MCQ accuracy: {sft_acc * 100:.1f}%")

# ── DPO accuracy ──────────────────────────────────────────────────────────
# model is already the DPO model (still in memory from training)
model.eval()
dpo_acc = evaluate_mcq_accuracy(
    model=model, tokenizer=med_tok, device=device,
    examples=test_examples, max_examples=TEST_SIZE, verbose=False,
)
print(f"DPO MCQ accuracy: {dpo_acc * 100:.1f}%")

print()
print("=" * 45)
print(f"{'Model':<20} {'MCQ Accuracy':>15}")
print("-" * 38)
print(f"{'SFT':<20} {sft_acc * 100:>14.1f}%")
print(f"{'DPO':<20} {dpo_acc * 100:>14.1f}%")
delta = (dpo_acc - sft_acc) * 100
print(f"{'Δ (DPO - SFT)':<20} {delta:>+14.1f} pp")
print("=" * 45)

## Next Steps

The DPO model is saved at `cfg.MED_DPO_FINAL_CKPT`.

- **Notebook 16** — Upload the tokenizer, SFT weights, DPO weights, and a model card
  to HuggingFace Hub so the model can be used and reproduced.

Record the MCQ accuracy numbers from this notebook in the model card (notebook 16, section 6).